DATASET ANALYSIS
======================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from wordcloud import WordCloud
from matplotlib import font_manager
from collections import Counter

# Load spaCy model 
nlp = spacy.load("en_core_web_sm")


# 1. LOAD DATA ---------------------------------

train_path = "train.csv"
df = pd.read_csv(train_path)

print("\n🔹 Dataset columns:", df.columns.tolist())
print("\n🔹 Dataset head:")
print(df.head())

# Detect text column
possible_text_cols = ["text", "pub_title", "dataset_title", "cleaned_label"]
text_col = next((col for col in possible_text_cols if col in df.columns), None)

if not text_col:
    raise ValueError(f"No text column found. Available columns: {df.columns.tolist()}")

print(f"\n✔ Using text column: {text_col}")


# 2. CLEAN TEXT  ---------------------------------

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

df["cleaned_text"] = df[text_col].apply(clean_text)

print("\n🔹 Cleaned text preview:")
print(df[["cleaned_text"]].head())


# 3. SIMPLE ENTITY EXTRACTION ---------------------------------

def extract_simple_entities(text):
    words = text.split()
    # Capitalized words removed earlier, so fallback entities = keywords
    return [w for w in words if len(w) > 8]

df["entities"] = df["cleaned_text"].apply(extract_simple_entities)

print("\n🔹 Entities preview:")
print(df[["entities"]].head())


# 4. WORD FREQUENCY ---------------------------------

all_words = " ".join(df["cleaned_text"]).split()
word_freq = Counter(all_words).most_common(30)

print("\n🔹 Top 10 most common words:")
print(word_freq[:10])

# Plot frequency
plt.figure(figsize=(12, 6))
words, counts = zip(*word_freq)
plt.bar(words, counts)
plt.xticks(rotation=90)
plt.title("Top 30 Most Common Words")
plt.show()


# 5. WORDCLOUD WITH SAFE TTF FONT ---------------------------------

try:
    font_path = font_manager.findfont("DejaVu Sans")
    wc = WordCloud(
        width=900,
        height=400,
        background_color="white",
        font_path=font_path
    ).generate(" ".join(all_words))

    plt.figure(figsize=(14, 7))
    plt.imshow(wc)
    plt.axis("off")
    plt.show()

except Exception as e:
    print("⚠ WordCloud could not be generated:", e)


# 6. DOCUMENT LENGTH ANALYSIS ---------------------------------

df["doc_length"] = df["cleaned_text"].apply(lambda x: len(x.split()))

plt.figure(figsize=(10, 5))
sns.histplot(df["doc_length"], bins=50)
plt.title("Document Length Distribution")
plt.show()


# 7. SUMMARY + SAVE CSV ---------------------------------

total_rows = len(df)
avg_length = df["doc_length"].mean()
top_words = word_freq[:10]

print("\n-------------------- SUMMARY --------------------")
print("Total rows:", total_rows)
print("Average doc length:", avg_length)
print("Most common words:", top_words)
print("--------------------------------------------------")

# Save cleaned file in same directory
train_dir = os.path.dirname(os.path.abspath(train_path))
output_csv = os.path.join(train_dir, "cleaned_dataset.csv")

df.to_csv(output_csv, index=False)
print(f"\n✔ Cleaned dataset exported as: {output_csv}")




NameError: name 'spacy' is not defined